# Derivative thresholds for DDS

* Read noise streams file and calculates derivative using DDS with several combinations of window and offset    
* Calculates RMS of noise derivative for each (w,o) combination


In [ ]:
import matplotlib.pyplot as plt
from astropy.io import fits
import numpy as np
from subprocess import run
%matplotlib widget

In [ ]:
windows = [0, 1, 2, 3, 4, 5, 6]
offsets = [0, 1, 2, 3, 4, 5, 6]

In [ ]:
def mean_derivative(derivative, window, offset=1):
    """mean_di[i] = 1/window * sum_{j=1}^{window} derivative[i-j-offset]"""
    n = len(derivative)
    mean_di = np.zeros(n, dtype=float)
    for i in range(window + offset, n):
        mean_di[i] = (1.0 / window) * sum(derivative[i - j - offset] for j in range(1, window + 1))
    return mean_di

In [ ]:
noise_file = "/dataj6/ceballos/INSTRUMEN/EURECA/ERESOL/CEASaclay/May2025_v5_v20250621/forNoise_50x30.fits"
kernel_coeficients = [1, 1, -1, -1]  # 1st derivative kernel
start_sample_for_kernel1 = 1 #[i=1, i=0, i=-1 , i=-2] # CEA/Saclay
lib_sirena = "/dataj6/ceballos/INSTRUMEN/EURECA/ERESOL/CEASaclay/May2025_v5_v20250621/optimal_filters_6keV_50x30.fits"
xml_xifusim = "config_xifu_50x30_v5_20250621.xml"


## Read Noise file

In [ ]:
with fits.open(noise_file) as hdulist:
    tclock = 7.68E-6
    sampling_rate = 1.0 / tclock  # Hz
    records = hdulist["TESRECORDS"].data
    record_stream = records["ADC"]


## Calculate derivative using DDS and baseline derivative (DDS with w=0)    

```d = 1* s_{i+1} + 1 * s_{i} + (-1)*s_{i-1} + (-1)*s_{i-2}```      # [1,1,-1,-1] DDS with w=0;o=0  (baseline)

In [ ]:
# create a numpy array to store a list of kernel coefficients for each combination of window and offset
kernel_coefficients_list = np.zeros((len(windows), len(offsets), 20), dtype=float)

In [ ]:
def derive_new_kernel(kernel_coeficients, start_sample_for_kernel1, window, offset=1):
    """DDS kernel for dw_i = d_i - mean_derivative(d, window, offset)[i],
    expressed directly in terms of the noise stream s.
    Returns {relative_offset: coefficient} such that
    dw_i = sum(c * s[i + r] for r, c in new_kernel.items())"""
    base = {}
    for k, c in enumerate(kernel_coeficients):
        r = start_sample_for_kernel1 - k
        base[r] = base.get(r, 0.0) + c

    new_kernel = dict(base)  # d_i term
    if window > 0:
        for j in range(1, window + 1):
            shift = -j - offset
            for r, c in base.items():
                m = r + shift
                new_kernel[m] = new_kernel.get(m, 0.0) - c / window

    return {m: c for m, c in new_kernel.items() if c != 0.0}


def derive_new_kernel_array(kernel_coeficients, start_sample_for_kernel1, window, offset, array_length=20):
    """Same as derive_new_kernel, but returned as a coefficient array indexed by
    k = start_sample_for_kernel1 - r, so it can be used directly as
    sum(arr[k] * record_stream[i + start_sample_for_kernel1 - k] for k in range(array_length))"""
    kernel = derive_new_kernel(kernel_coeficients, start_sample_for_kernel1, window, offset)
    arr = np.zeros(array_length, dtype=float)
    for r, c in kernel.items():
        k = start_sample_for_kernel1 - r
        arr[k] = c
    return arr


for iw, window in enumerate(windows):
    for io, offset in enumerate(offsets):
        if window == 0 and offset > 0:
            continue  # skip this case since it is not defined
        kernel_coefficients_list[iw, io, :] = derive_new_kernel_array(
            kernel_coeficients, start_sample_for_kernel1, window, offset
        )

In [ ]:
# do some testing to check that the kernel coefficients are correct
print("Testing kernel coefficients for window=2, offset=1")
print("kernel coefficients:", kernel_coefficients_list[2, 1, :])

In [ ]:
def rms_from_kernel(record_stream, kernel_array, start_sample_for_kernel1):
    """Vectorized RMS of the derivative defined by kernel_array (as produced by
    derive_new_kernel_array), applied to every record in record_stream (n_records, n_samples).
    Returns an array of shape (n_records,)."""
    nz = np.nonzero(kernel_array)[0]
    rs = start_sample_for_kernel1 - nz  # relative offsets for each nonzero tap
    r_min, r_max = rs.min(), rs.max()

    n_samples = record_stream.shape[1]
    j_start = -r_min
    j_end = n_samples - r_max  # exclusive

    deriv = np.zeros((record_stream.shape[0], j_end - j_start), dtype=float)
    for k, r in zip(nz, rs):
        deriv += kernel_array[k] * record_stream[:, j_start + r: j_end + r]

    return np.sqrt(np.mean(deriv ** 2, axis=1))


In [ ]:
# rms_derivative[iw, io, record] = RMS of the (window, offset) derivative for that record stream
rms_derivative = np.full((len(windows), len(offsets), record_stream.shape[0]), np.nan, dtype=float)
mean_rms_derivative = np.full((len(windows), len(offsets)), np.nan, dtype=float)
std_rms_derivative = np.full((len(windows), len(offsets)), np.nan, dtype=float)

for iw, window in enumerate(windows):
    for io, offset in enumerate(offsets):
        if window == 0 and offset > 0:
            continue  # skip this case since it is not defined
        rms_derivative[iw, io, :] = rms_from_kernel(
            record_stream, kernel_coefficients_list[iw, io, :], start_sample_for_kernel1
        )
        mean_rms_derivative[iw, io] = np.nanmean(rms_derivative[iw, io, :])
        std_rms_derivative[iw, io] = np.nanstd(rms_derivative[iw, io, :])

In [ ]:
for window in windows:
    for offset in offsets:
        print(f"Window: {window}, Offset: {offset}, Mean RMS: {mean_rms_derivative[window, offset]:.6f}, Std RMS: {std_rms_derivative[window, offset]:.6f}")

# SIRENA reconstruction of NOISE streams   

* Use different detection thresholds: [1, 1.5, 2., 2.5, 3., 3.5, 4., 4.5, 5.]*sigma
* For each combination of window/offset, sigma is different   
* Reconstruct every noise stream record and calculate number of detections (they will be False detections)

Logic:   

```
for each win:
    for each off:
        get mean_rms(win,off)
        for each th in [1, 1.5, 2., 2.5, 3., 3.5, 4., 4.5, 5.]*mean_rms
            run SIRENA (win, off, th) -> get detections=false_detections[win,off,nsigmas]


In [ ]:
th_sigmas = [1, 1.5, 2., 2.5, 3., 3.5, 4., 4.5, 5.]
SD = 2
SU = 3

In [ ]:
for iw in range(len(windows)):
    window = windows[iw]
    for io in range(len(offsets)):
        offset = offsets[io]
        if window == 0 and offset > 0:
            continue  # skip this case since it is not defined
        for th_sigma in th_sigmas:
            threshold = th_sigma * mean_rms_derivative[iw, io]
            print(f"Window: {window}, Offset: {offset}, Threshold is {th_sigma} sigmas")
            falses_file = f"./analysis_pairs/falses_window{window}_offset{offset}_th{th_sigma}sigmas.fits"
            comm = (f"tesrecons Recordfile={noise_file} "
                        f" TesEventFile={falses_file}"
                        f" LibraryFile={lib_sirena}"
                        f" XMLFile={xml_xifusim}"
                        f" clobber=yes"
                        f" EnergyMethod=OPTFILT"
                        f" OFStrategy=BYGRADE"
                        f" filtEeV=6000"
                        f" OFNoise=NSD"
                        f" samplesDown={SD}"   #changed for new smoothed derivative (4 samples)
                        f" samplesUp={SU}"
                        f" threshold={threshold}"
                        f" windowSize={window}"
                        f" offset={offset}"
                    )
            #aux.vprint(f"Running {comm}")
            output_tesrecons = run(comm, shell=True, capture_output=True)
            assert output_tesrecons.returncode == 0, f"tesrecons failed to run:{comm}; \nReason: {output_tesrecons.stderr.decode()}"

## Save a file with detection results